In [3]:
import os
import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import Dataset, DataLoader
from nnAudio.features.gammatone import Gammatonegram
from tqdm import tqdm

In [ ]:
SR = 8000               # 아두이노 sampling rate (8kHz)
N_FFT = 512             # 감마톤 필터 계산 시 사용하는 윈도우 크기
N_BINS = 64             # 감마톤 필터 채널 개수
HOP_LENGTH = 128        # 윈도우를 128샘플씩 이동
DURATION_SEC = 2        # 학습 시 오디오를 2초 단위로 잘라서 사용
BATCH_SIZE = 16         # 배치 크기
EPOCHS = 30             # 학습 에포크 수
LR = 1e-3               # 학습률
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class GammatoneDataset(Dataset):
    def __init__(self, clean_paths, noise_paths):
        self.clean_paths = clean_paths
        self.noise_paths = noise_paths
        self.max_len = SR * DURATION_SEC

        self.gt = Gammatonegram(
            sr=SR,
            n_fft=N_FFT,
            n_bins=N_BINS,
            hop_length=HOP_LENGTH,
            fmin=100.0,
            fmax=4000.0
        ).to(DEVICE)

    def __len__(self):
        return len(self.clean_paths)

    def load_wav(self, path):
        wave, sr = torchaudio.load(path)
        if wave.shape[0] > 1:  # 스테레오면 모노로
            wave = torch.mean(wave, dim=0, keepdim=True)
        if sr != SR:
            wave = torchaudio.functional.resample(wave, sr, SR)
        return wave

    def fix_len(self, wave):
        if wave.shape[1] > self.max_len:
            return wave[:, :self.max_len]
        return torch.nn.functional.pad(wave, (0, self.max_len - wave.shape[1]))

    def __getitem__(self, idx):
        clean = self.fix_len(self.load_wav(self.clean_paths[idx]))

        # 노이즈는 랜덤으로 하나 선택
        rand_idx = torch.randint(0, len(self.noise_paths), (1,)).item()
        noise = self.fix_len(self.load_wav(self.noise_paths[rand_idx]))

        # -5 ~ 5dB 사이 랜덤 SNR로 mix
        snr = torch.FloatTensor(1).uniform_(-5, 5).item()
        p_clean = clean.pow(2).mean() # 순수 사이렌 신호의 평균 power
        p_noise = noise.pow(2).mean() # noise 신호의 평균 power
        scale = torch.sqrt(p_clean / (p_noise * (10 ** (snr / 10)) + 1e-10)) # 소음 신호의 가중치 (얼마나 소음 비중을 높게 할건지)
        noise_scaled = scale * noise
        mixed = clean + noise_scaled

        # 감마톤 스펙트로그램
        with torch.no_grad():
            c_gt = self.gt(clean.to(DEVICE))
            n_gt = self.gt(noise_scaled.to(DEVICE))
            m_gt = self.gt(mixed.to(DEVICE))

        # IRM 계산
        irm = torch.sqrt((c_gt ** 2) / (c_gt ** 2 + n_gt ** 2 + 1e-8))

        # (Time, Bins) 형태로 변환
        m_gt_2d = m_gt[0] #shape : (Time_Frames, Filter_Bins)
        irm_2d = irm[0]

        X = m_gt_2d.transpose(0, 1)
        Y = irm_2d.transpose(0, 1)

        return X.cpu(), Y.cpu()

In [ ]:
class SirenGRU(nn.Module):
    def __init__(self, input_size=N_BINS, hidden_size=128, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False  # 실시간 처리라 단방향
        )
        self.fc = nn.Linear(hidden_size, input_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.gru(x)
        return self.sigmoid(self.fc(out))
